# Capture Capacity Metrics App data to a lakehouse

This notebook connects to the Fabric Capacity Metrics App semantic model with Semantic Link, extracts configured tables or DAX query results, and writes them into the attached lakehouse as Delta tables.

Attach the target lakehouse before running this notebook. Schedule it from a Fabric pipeline or notebook schedule to keep the lakehouse updated automatically.

In [ ]:
# Parameters
# Workspace that contains the Capacity Metrics App semantic model. Leave blank to use the current workspace.
semantic_model_workspace = ""

# Common model names are "Fabric Capacity Metrics" or a tenant-specific renamed copy.
semantic_model_name = "Fabric Capacity Metrics"

# Leave empty to discover visible tables from the semantic model and export each table.
# Example: ["Capacities", "Items", "TimePointBackgroundDetail"]
tables_to_export = []

# Optional named DAX queries. When provided, these are exported in addition to tables_to_export.
# The dictionary key becomes the output table suffix.
# Example: {"cu_by_day": "EVALUATE SUMMARIZECOLUMNS('Date'[Date], \"CU\", [CU])"}
dax_queries = {}

# Output settings. Tables are written to the attached lakehouse.
# Use dbo for the lakehouse default schema, or set another schema name.
target_schema = "dbo"
target_table_prefix = "capacity_metrics"
write_mode = "append"  # append or overwrite
fail_on_query_error = True


In [ ]:
from datetime import datetime, timezone
import re
import uuid

import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql import types as T

try:
    import sempy.fabric as fabric
except ImportError as exc:
    raise ImportError(
        "Semantic Link is required. Run this notebook in Microsoft Fabric with semantic-link available."
    ) from exc

if write_mode not in {"append", "overwrite"}:
    raise ValueError("write_mode must be 'append' or 'overwrite'.")

target_schema = target_schema.strip() or "dbo"
if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", target_schema):
    raise ValueError("target_schema must contain only letters, numbers, and underscores, and cannot start with a number.")

run_id = str(uuid.uuid4())
capture_timestamp_utc = datetime.now(timezone.utc)
workspace_arg = semantic_model_workspace.strip() or None

print(f"Run ID: {run_id}")
print(f"Capture timestamp UTC: {capture_timestamp_utc.isoformat()}")
print(f"Semantic model: {semantic_model_name}")
print(f"Semantic model workspace: {workspace_arg or 'current workspace'}")
print(f"Target schema: {target_schema}")


In [ ]:
def sanitize_identifier(value: str) -> str:
    cleaned = re.sub(r"[^A-Za-z0-9_]+", "_", str(value).strip()).strip("_").lower()
    if not cleaned:
        cleaned = "unnamed"
    if cleaned[0].isdigit():
        cleaned = f"_{cleaned}"
    return cleaned


def normalize_columns(pdf: pd.DataFrame) -> pd.DataFrame:
    normalized = []
    seen = {}

    for column in pdf.columns:
        name = str(column).replace("[", "_").replace("]", "")
        name = sanitize_identifier(name)
        count = seen.get(name, 0)
        seen[name] = count + 1
        normalized.append(name if count == 0 else f"{name}_{count + 1}")

    result = pdf.copy()
    result.columns = normalized
    return result


def pandas_to_spark_schema(pdf: pd.DataFrame) -> T.StructType:
    fields = []
    for column_name, dtype in pdf.dtypes.items():
        if pd.api.types.is_bool_dtype(dtype):
            data_type = T.BooleanType()
        elif pd.api.types.is_integer_dtype(dtype):
            data_type = T.LongType()
        elif pd.api.types.is_float_dtype(dtype):
            data_type = T.DoubleType()
        elif pd.api.types.is_datetime64_any_dtype(dtype):
            data_type = T.TimestampType()
        else:
            data_type = T.StringType()
        fields.append(T.StructField(column_name, data_type, True))
    return T.StructType(fields)


def qualified_table_name(table_name: str) -> str:
    return f"`{target_schema}`.`{table_name}`"


def ensure_delta_table(table_name: str, dataframe) -> str:
    qualified_name = qualified_table_name(table_name)
    if not spark.catalog.tableExists(f"{target_schema}.{table_name}"):
        dataframe.limit(0).write.format("delta").mode("ignore").saveAsTable(qualified_name)
        print(f"Created table {target_schema}.{table_name}")
    return qualified_name


def quote_dax_table_name(table_name: str) -> str:
    return "'" + table_name.replace("'", "''") + "'"


def evaluate_dax_query(dax_query: str) -> pd.DataFrame:
    return fabric.evaluate_dax(
        dataset=semantic_model_name,
        dax_string=dax_query,
        workspace=workspace_arg,
    )


def discover_visible_tables() -> list[str]:
    try:
        table_df = fabric.list_tables(dataset=semantic_model_name, workspace=workspace_arg)
    except Exception:
        table_df = evaluate_dax_query("EVALUATE INFO.VIEW.TABLES()")

    if table_df.empty:
        return []

    column_lookup = {str(column).lower(): column for column in table_df.columns}
    name_column = column_lookup.get("name") or column_lookup.get("table") or column_lookup.get("table_name")
    hidden_column = column_lookup.get("ishidden") or column_lookup.get("is_hidden") or column_lookup.get("hidden")

    if name_column is None:
        raise RuntimeError(f"Could not identify a table-name column in semantic model metadata: {list(table_df.columns)}")

    if hidden_column is not None:
        table_df = table_df[~table_df[hidden_column].fillna(False).astype(bool)]

    names = [str(name) for name in table_df[name_column].dropna().tolist()]
    return [name for name in names if not name.startswith("LocalDateTable_")]


In [ ]:
exports = {}

selected_tables = tables_to_export or discover_visible_tables()
for table_name in selected_tables:
    output_name = sanitize_identifier(table_name)
    exports[output_name] = f"EVALUATE {quote_dax_table_name(table_name)}"

for output_name, dax_query in dax_queries.items():
    exports[sanitize_identifier(output_name)] = dax_query

if not exports:
    raise RuntimeError(
        "No tables or DAX queries were selected. Set tables_to_export or dax_queries, "
        "or confirm the semantic model can be discovered with Semantic Link."
    )

spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{target_schema}`")
print(f"Schema ready: {target_schema}")

print("Planned exports:")
for output_name in exports:
    print(f"- {target_schema}.{target_table_prefix}_{output_name}")


In [ ]:
run_log = []
failures = []

for output_name, dax_query in exports.items():
    lakehouse_table = f"{target_table_prefix}_{output_name}"
    started_at = datetime.now(timezone.utc)

    try:
        pdf = evaluate_dax_query(dax_query)
        row_count = len(pdf.index)

        pdf = normalize_columns(pdf)
        pdf["_capture_run_id"] = run_id
        pdf["_capture_timestamp_utc"] = capture_timestamp_utc
        pdf["_source_semantic_model"] = semantic_model_name
        pdf["_source_workspace"] = workspace_arg or "current workspace"
        pdf["_source_query_name"] = output_name

        if row_count > 0:
            spark_df = spark.createDataFrame(pdf)
        else:
            spark_df = spark.createDataFrame([], schema=pandas_to_spark_schema(pdf))
        spark_df = spark_df.withColumn("_capture_timestamp_utc", F.to_timestamp(F.col("_capture_timestamp_utc")))
        qualified_lakehouse_table = ensure_delta_table(lakehouse_table, spark_df)

        if row_count > 0:
            (
                spark_df.write
                .format("delta")
                .mode(write_mode)
                .option("mergeSchema", "true")
                .saveAsTable(qualified_lakehouse_table)
            )

        completed_at = datetime.now(timezone.utc)
        run_log.append({
            "capture_run_id": run_id,
            "capture_timestamp_utc": capture_timestamp_utc,
            "query_name": output_name,
            "target_table": f"{target_schema}.{lakehouse_table}",
            "status": "succeeded",
            "row_count": row_count,
            "started_at_utc": started_at,
            "completed_at_utc": completed_at,
            "error_message": None,
        })
        print(f"Wrote {row_count:,} rows to {target_schema}.{lakehouse_table}")

    except Exception as exc:
        completed_at = datetime.now(timezone.utc)
        error_message = str(exc)
        failures.append((output_name, error_message))
        run_log.append({
            "capture_run_id": run_id,
            "capture_timestamp_utc": capture_timestamp_utc,
            "query_name": output_name,
            "target_table": f"{target_schema}.{lakehouse_table}",
            "status": "failed",
            "row_count": 0,
            "started_at_utc": started_at,
            "completed_at_utc": completed_at,
            "error_message": error_message,
        })
        print(f"Failed export {output_name}: {error_message}")

run_log_df = spark.createDataFrame(pd.DataFrame(run_log))
for column_name in ["capture_timestamp_utc", "started_at_utc", "completed_at_utc"]:
    run_log_df = run_log_df.withColumn(column_name, F.to_timestamp(F.col(column_name)))

run_log_table = f"{target_table_prefix}_capture_run_log"
qualified_run_log_table = ensure_delta_table(run_log_table, run_log_df)

(
    run_log_df.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(qualified_run_log_table)
)

if failures and fail_on_query_error:
    failure_text = "; ".join([f"{name}: {message}" for name, message in failures])
    raise RuntimeError(f"One or more Capacity Metrics exports failed: {failure_text}")

print(f"Capture complete. Run log written to {target_schema}.{run_log_table}")


## Scheduling notes

For automatic capture, add this notebook to a Fabric pipeline or notebook schedule. Use pipeline parameters to set `semantic_model_workspace`, `semantic_model_name`, `tables_to_export`, `target_schema`, and `write_mode`. The default schema is `dbo`. If another schema is configured, the notebook creates it before creating its Delta tables. Keep `write_mode` set to `append` for normal scheduled runs.